# KAKEN 大型科研費までの道のり — Colab版

KAKEN「**研究者をさがす**」からダウンロードしたJSONを読み込み、所属機関の研究者が
**最初の科研費から大型科研費（基盤研究(B)以上）に到達するまでの年数**を集計し、
レポート・ヒストグラム・相対年ガントチャートのPDFを作ります。**appidは不要**です。

## 使い方（上のセルから順に実行）
1. 環境の準備（ライブラリと日本語フォントの導入）
2. KAKENからJSONを取得してアップロード
3. 機関名を入力
4. 集計＆PDF生成
5. PDFをダウンロード

### KAKENでのJSONの取り方（appid不要）
1. [KAKEN 研究者をさがす](https://nrid.nii.ac.jp/ja/) を開く
2. 詳細検索で **研究機関＝自分の機関名** で検索
3. 検索結果画面で **Select All → Export in JSON** を実行してJSONを保存

> **データの扱い**: アップロードしたJSONは、あなたが今開いているこのColabセッション
> （Googleのクラウド上の一時環境）でのみ処理されます。KAKENの公開データですが、氏名・
> 研究者番号を含みます。セッションを閉じれば消えます。手元だけで処理したい場合は、
> リポジトリをローカルにcloneしてPythonで実行する方法もあります（READMEを参照）。

> **1万件の上限**: KAKENの研究者検索エクスポートは1回あたり1万件までです。研究者数が
> それを超える大規模機関では、検索を分割してエクスポートする必要があります。

## 1. 環境の準備

ライブラリと日本語フォント（IPAゴシック）を入れ、コードを取得します。

In [ ]:
# 日本語フォント（IPAゴシック）とPDF変換ツールを導入
!apt-get -qq -y install fonts-ipafont-gothic poppler-utils > /dev/null
!pip -q install matplotlib pdf2image > /dev/null

# 入れたIPAフォントを matplotlib に登録（中国語字形になるNoto CJKは使わない）
import glob, matplotlib.font_manager as fm, matplotlib.pyplot as plt
ipa = glob.glob('/usr/share/fonts/**/ipag*.ttf', recursive=True)
for path in ipa:
    fm.fontManager.addfont(path)
plt.rcParams['font.family'] = 'IPAGothic'
print('日本語フォント:', 'OK' if ipa else '見つかりません（描画が豆腐□になる場合はランタイム再起動）')

# コード一式を取得
import os
if not os.path.exists('kaken-summary'):
    !git clone -q https://github.com/takayuki1997/kaken-summary.git
%cd kaken-summary
print('準備完了')

## 2. JSONをアップロード

KAKENからダウンロードした研究者JSONを選んでアップロードします。

> 大きいファイル（数百MB）はアップロードに時間がかかります。完了まで待ってください。

In [ ]:
from google.colab import files
uploaded = files.upload()   # KAKENの研究者JSONを選択
assert uploaded, 'ファイルが選択されていません'
UPLOADED_JSON = list(uploaded.keys())[0]
print('アップロード:', UPLOADED_JSON)

## 3. 機関名を入力

レポートのタイトルなどに使う機関名を入力してください（母集団はJSONの中身で決まります）。

In [ ]:
INST_NAME = "\u25cb\u25cb\u5927\u5b66"  #@param {type:"string"}
KEY = 'target'

import shutil, os, kaken_inst
kaken_inst.INSTITUTIONS[KEY] = INST_NAME       # 機関キーを実行時に登録
os.makedirs(f'data/{KEY}', exist_ok=True)
shutil.move(UPLOADED_JSON, f'data/{KEY}/researchers.json')
print(f'{INST_NAME} のデータを配置しました')

## 4. 集計＆PDF生成

3種類のPDFを作ります：
- `kaken_report_target.pdf` … A4縦1枚レポート（氏名なし）
- `kaken_stepup_hist_target.pdf` … 所要年数ヒストグラム（氏名なし）
- `kaken_stepup_gantt_target.pdf` … 相対年アラインのガント（**氏名あり**）

In [ ]:
import sys
sys.argv = ['', KEY]

import kaken_stepup, kaken_report, kaken_stepup_gantt

kaken_stepup.main()
kaken_report.main()
kaken_stepup_gantt.main()

### レポートを画面で確認

In [ ]:
from pdf2image import convert_from_path
from IPython.display import display
img = convert_from_path(f'output/kaken_report_{KEY}.pdf', dpi=120)[0]
display(img)

## 5. PDFをダウンロード

In [ ]:
from google.colab import files
for f in ['kaken_report', 'kaken_stepup_hist', 'kaken_stepup_gantt']:
    files.download(f'output/{f}_{KEY}.pdf')